# 02 — Baseline Xception Model Training

This notebook trains a baseline Xception model for four-class brain MRI
tumor classification.

**Classes**
- Glioma
- Meningioma
- No Tumor
- Pituitary

The Xception backbone is initialized with ImageNet weights and kept frozen
during baseline training. Fine-tuning is addressed separately.


# 1. Import Needed Libraries

In [2]:
from pathlib import Path
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception



# 2. Reproducibility & Path Configuration

In [3]:
SEED = 42

BATCH_SIZE = 32
TEST_BATCH_SIZE = 16

IMG_SIZE = (299, 299)
IMAGE_SHAPE = (299, 299, 3)

EPOCHS = 10
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
]

PROJECT_ROOT = Path("..")

DATAFRAMES_DIR = PROJECT_ROOT / "results" / "dataframes"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "xception_baseline.keras"


In [4]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 3. Read DataFrames

In [5]:
train_df = pd.read_csv(DATAFRAMES_DIR / "train_dataframe.csv")
valid_df = pd.read_csv(DATAFRAMES_DIR / "validation_dataframe.csv")
test_df = pd.read_csv(DATAFRAMES_DIR / "test_dataframe.csv")

In [6]:
print(f"Training samples   : {len(train_df):,}")
print(f"Validation samples : {len(valid_df):,}")
print(f"Test samples       : {len(test_df):,}")


Training samples   : 5,600
Validation samples : 800
Test samples       : 800


In [7]:
required_columns = {"filepath", "label"}

for df_name, df in {
    "tr_df": train_df,
    "valid_df": valid_df,
    "ts_df": test_df,
}.items():
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{df_name} is missing required columns: {missing_columns}"
        )

print("All required DataFrame columns are available.")


All required DataFrame columns are available.


# 4. Data Generator Construction

In [8]:
train_datagen = ImageDataGenerator(
    rescale=1 / 255,
    rotation_range=10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    brightness_range=(0.85, 1.15),
    horizontal_flip=True,
)

test_datagen = ImageDataGenerator(
    rescale=1 / 255,
)

tr_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
)

valid_gen = test_datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=TEST_BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print("\nClass indices:")
print(tr_gen.class_indices)


Found 5600 validated image filenames belonging to 4 classes.
Found 800 validated image filenames belonging to 4 classes.
Found 800 validated image filenames belonging to 4 classes.

Class indices:
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


# 5. Model Architecture & Compilation

In [9]:
base_model = tf.keras.applications.Xception(
    include_top=False,
    weights="imagenet",
    input_shape=IMAGE_SHAPE,
    pooling="max",
)

base_model.trainable = True

model = Sequential(
    [
        base_model,
        Dropout(rate=0.30),
        Dense(128, activation="relu"),
        Dropout(rate=0.25),
        Dense(len(CLASS_NAMES), activation="softmax"),
    ],
    name="xception_brain_tumor_classifier",
)


model.compile(
    optimizer=Adamax(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall"),
    ],
)

model.summary()


Model: "xception_brain_tumor_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 2048)           │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,124,268 (80.58 MB)

 Trainable params: 21,069,740 (80.37 MB)

 Non-trainable params: 54,528 (213.00 KB)

# 6. Callbacks and Model Training

In [10]:
training_callbacks = [
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [11]:
history = model.fit(
    tr_gen,
    validation_data=valid_gen,
    epochs=EPOCHS,
    callbacks=training_callbacks,
)


Epoch 1/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.7307 - loss: 0.7108 - precision: 0.8048 - recall: 0.6295 
Epoch 1: val_loss improved from None to 0.60352, saving model to ../models/xception_baseline.keras

Epoch 1: finished saving model to ../models/xception_baseline.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 2610s 15s/step - accuracy: 0.7307 - loss: 0.7108 - precision: 0.8048 - recall: 0.6295 - val_accuracy: 0.7475 - val_loss: 0.6035 - val_precision: 0.7903 - val_recall: 0.7163 - learning_rate: 1.0000e-04
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.8961 - loss: 0.2854 - precision: 0.9111 - recall: 0.8823 
Epoch 2: val_loss improved from 0.60352 to 0.49259, saving model to ../models/xception_baseline.keras

Epoch 2: finished saving model to ../models/xception_baseline.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 2488s 14s/step - accuracy: 0.8961 - loss: 0.2854 - precision: 0.9111 - recall: 0.8823 - val_accuracy: 0.8000 - val_loss: 0.4926 - val_precision: 0.8294 - 

In [12]:
import pickle
HISTORY_PATH = RESULTS_DIR / "baseline_training_history.pkl"

with open(HISTORY_PATH, "wb") as file:
    pickle.dump(history.history, file)

print(f"Best model already saved to: {MODEL_PATH}")
print(f"Training history saved to: {HISTORY_PATH}")

Best model already saved to: ../models/xception_baseline.keras
Training history saved to: ../results/baseline_training_history.pkl
